# Spotify Hit Prediction - Preprocessing

L'objectif de cette étape est de préparer les données pour la modélisation en appliquant les transformations nécessaires:

- Nettoyage des données  
- Sélection des variables  
- Feature engineering (création et transformation de variables)  
- Encodage des variables catégorielles  
- Normalisation : mise à l’échelle des variables numériques  


In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore")

Les données sont chargées depuis Hugging Face puis converties en DataFrame pandas pour les manipulations.

In [2]:
dataset = load_dataset("Faizasb/spotify-tracks-dataset")
df = dataset["train"].to_pandas()
print(f"Taille du dataset d'origine: {df.shape}")
df.head()

Taille du dataset d'origine: (114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 3. Création de la variable cible

La variable `is_hit` est créée à partir de la variable `popularity`.  
Un morceau est considéré comme un hit s’il fait partie des 20% les plus populaires.

In [3]:
seuil = df["popularity"].quantile(0.8)
df["is_hit"] = df["popularity"] >= seuil
print(df["is_hit"].value_counts())

is_hit
False    90570
True     23430
Name: count, dtype: int64


## 4. Séparation des variables explicatives et de la cible

La variable cible `is_hit` est séparée des variables explicatives afin de préparer les données pour la modélisation.

In [4]:
X = df.drop("is_hit", axis=1)
y = df["is_hit"]

print(X.shape)
print(y.shape)

(114000, 21)
(114000,)


## 5. Séparation en jeu d'entraînement et de test (Train set et Test set)

Les données sont séparées en un jeu d'entraînement et un jeu de test afin d'évaluer les performances du modèle sur des données jamais vues.

La stratification est utilisée pour conserver la proportion de hits dans les deux jeux de données (environ 20% de hits dans chaque ensemble).

- Train set : 80% des données
- Test set : 20% des données

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (91200, 21)
Test: (22800, 21)


## 6. Sélection des variables

Certaines variables sont supprimées avant la modélisation :

- `Unnamed: 0` : index technique sans valeur informative.
- Variables d’identification (`track_id`, `track_name`, `album_name`) : elles n’apportent pas d’information utile.
- `artists` : risque de surapprentissage (le modèle pourrait mémoriser les artistes).
- `popularity` : utilisée pour créer la cible -->  risque de fuite d’information.
- `mode`, `key`, `time_signature` : variables peu informatives d’après l’EDA.

In [6]:
cols_to_drop = [
    "Unnamed: 0",
    "track_id",
    "track_name",
    "album_name",
    "artists",
    "popularity",
    "mode",
    "key",
    "time_signature",
]

X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (91200, 12)
Test: (22800, 12)


## 7. Feature engineering : regroupement des genres

La variable `track_genre` contient un grand nombre de catégories (114), ce qui peut compliquer la modélisation.

Afin de réduire la complexité et améliorer la généralisation du modèle, les genres sont regroupés en grandes catégories musicales cohérentes.

In [7]:
def simplify_genre(genre):
    match genre:
        # ROCK
        case g if g in {
            "rock",
            "alt-rock",
            "alternative",
            "indie",
            "punk",
            "punk-rock",
            "grunge",
            "hard-rock",
            "emo",
            "goth",
            "psych-rock",
            "rock-n-roll",
            "rockabilly",
            "j-rock",
            "british",
        }:
            return "rock"

        # METAL
        case g if g in {
            "metal",
            "heavy-metal",
            "black-metal",
            "death-metal",
            "metalcore",
            "grindcore",
            "hardcore",
            "industrial",
        }:
            return "metal"

        # ELECTRONIC
        case g if g in {
            "electronic",
            "edm",
            "electro",
            "house",
            "deep-house",
            "chicago-house",
            "detroit-techno",
            "minimal-techno",
            "techno",
            "trance",
            "progressive-house",
            "hardstyle",
            "dubstep",
            "drum-and-bass",
            "breakbeat",
            "idm",
            "garage",
            "club",
            "dance",
            "disco",
            "trip-hop",
        }:
            return "electronic"

        # POP
        case g if g in {
            "pop",
            "synth-pop",
            "power-pop",
            "pop-film",
            "indie-pop",
            "k-pop",
            "j-pop",
            "j-idol",
            "j-dance",
            "cantopop",
            "mandopop",
        }:
            return "pop"

        # URBAN
        case g if g in {"hip-hop", "r-n-b"}:
            return "urban"

        # JAZZ / SOUL
        case g if g in {"jazz", "soul", "funk", "blues", "gospel", "groove"}:
            return "jazz_soul"

        # CLASSICAL
        case g if g in {"classical", "opera", "piano", "new-age"}:
            return "classical_instrumental"

        # FOLK / COUNTRY
        case g if g in {
            "folk",
            "country",
            "bluegrass",
            "honky-tonk",
            "acoustic",
            "singer-songwriter",
            "songwriter",
            "guitar",
        }:
            return "folk_country"

        # LATIN / WORLD
        case g if g in {
            "latin",
            "latino",
            "reggaeton",
            "salsa",
            "samba",
            "forro",
            "mpb",
            "pagode",
            "sertanejo",
            "brazil",
            "tango",
            "spanish",
            "french",
            "german",
            "swedish",
            "turkish",
            "indian",
            "iranian",
            "malay",
            "world-music",
            "afrobeat",
        }:
            return "latin_world"

        # REGGAE
        case g if g in {"reggae", "dancehall", "ska", "dub"}:
            return "reggae_caribbean"

        # AMBIENT / MOOD
        case g if g in {
            "ambient",
            "chill",
            "sleep",
            "study",
            "happy",
            "sad",
            "party",
            "romance",
        }:
            return "ambient_mood"

        # MEDIA / FUNCTIONAL
        case g if g in {"kids", "children", "anime", "disney", "comedy", "show-tunes"}:
            return "functional_media"

        # OTHER
        case _:
            return "other"

In [8]:
X_train["track_genre"] = X_train["track_genre"].apply(simplify_genre)
X_test["track_genre"] = X_test["track_genre"].apply(simplify_genre)

print(X_train["track_genre"].value_counts())
print()
print(
    f"On passe de {df['track_genre'].nunique()} genres musicaux à {X_train['track_genre'].nunique()} genres."
)

track_genre
latin_world               16823
electronic                16822
rock                      11951
pop                        8781
metal                      6408
ambient_mood               6368
folk_country               6337
functional_media           4852
jazz_soul                  4843
classical_instrumental     3219
reggae_caribbean           3157
urban                      1639
Name: count, dtype: int64

On passe de 114 genres musicaux à 12 genres.
